# Spins (Hund's rule) and automatic configuration ensembles

In [1]:
import numpy as np
from pyscf import gto
from f4b import fgto

## Ground-state spin by Hund's rule

Every fragment member gets its **ground-state spin** unless you give it. For an isolated fragment with $N$ electrons, `f4b` runs a small symmetric
calculation (the electrons of a partly filled group of degenerate orbitals are shared equally, so the density is spherical/cylindrical and the
degenerate orbitals are exactly degenerate), fills the orbitals in order of energy and puts the electrons of the last, partly filled degenerate group
parallel first ($N_\alpha - N_\beta = \min(k, 2d-k)$ for $k$ electrons in $d$ orbitals). `mol.shells(fragment, N)` shows what it found.

In [2]:
mol = fgto.M(atom='F 0 0 0; H 0 0 1.733', unit='bohr', basis='cc-pvdz', fragments={0: [0], 1: [1]},
             ensembles={'charge': {0: 8.7, 1: 1.3}})
print(mol)                     # F+ (8 e, triplet: 5 alpha + 3 beta) and F (9 e, doublet: 5 alpha + 4 beta); H- and H
print(mol.shells(0, 8))        # the 2p shell holds 4 electrons: 3 alpha, 1 beta

FragMole:
  Fragment 0: atoms=(0,) Z=9 members=[(5,3;w=0.3,flip=0), (5,4;w=0.7,flip=0)]
  Fragment 1: atoms=(1,) Z=1 members=[(1,0;w=0.7,flip=0), (1,1;w=0.3,flip=0)]
Shells of fragment 0 with 8 electrons (Hund: N_alpha=5, N_beta=3, 2S=2); orbitals in order of energy
  group  degeneracy  energy (Eh)   electrons   main AO components
      0           1    -24.96676       2/2     0 F 1s (1.00), 0 F 2s (-0.00)
      1           1     -1.73405       2/2     0 F 2s (0.52), 0 F 3s (0.48)
      2           3     -1.04803       4/6     0 F 2py (0.24), 0 F 2px (0.24)
      3           3      0.48436       0/6     0 F 3px (0.24), 0 F 3py (0.24)


The spin of the neutral member can be set with `fspin={fragment: 2S}` ($2S=N_\alpha-N_\beta$) and, for the advanced charge ensemble form, each
member's `'spins'`. `fcharge={fragment: charge}` is only needed for ions **without** charge ensemble (HeH$^+$).

In [3]:
atoms = 'O 0 0 0; O 0 0 2.3'
for fspin in (None, {0: 0, 1: 0}):
    m = fgto.M(atom=atoms, unit='bohr', basis='cc-pvdz', fragments={0: [0], 1: [1]}, fspin=fspin)
    print('fspin =', fspin, '->', [(mem.na, mem.nb) for f in m.frags for mem in f.members], ' (N_alpha, N_beta of each O atom)')

fspin = None -> [(5, 3), (5, 3)]  (N_alpha, N_beta of each O atom)
fspin = {0: 0, 1: 0} -> [(4, 4), (4, 4)]  (N_alpha, N_beta of each O atom)


## Configuration ensembles for degenerate shells

If the last group of orbitals is only partly filled, its occupations are degenerate: $C(d, n)$ determinants for $n$ electrons of one spin in $d$
orbitals. `ensembles={'config': [fragments]}` builds them. They are selected with population constraints on **holes** (`fill='holes'`, default, more accurate)
or on **occupied** orbitals (`fill='occupied'`). Shell orbitals are oriented along the Cartesian axes (atoms) or the molecular axis (linear molecules):
the p orbitals of an atom are $p_x, p_y, p_z$ and the $\pi$ orbitals of CO are $\pi_x$, $\pi_y$.

The number of states for typical fragments:

In [4]:
def count(atom, N, label):
    Z = int(sum(gto.mole.charge(a.split()[0]) for a in atom.split(';')))
    m = fgto.M(atom=atom, unit='bohr', basis='cc-pvdz', fragments={0: list(range(atom.count(';') + 1))}, charge=Z - N, spin=N % 2, fcharge={0: Z - N})
    ref = m.shells(0, N)
    na, nb = ref.hund()
    states = ref.states(na, nb)
    print(f"{label:6s} N={N:2d}  Hund: {na} alpha + {nb} beta (2S={na-nb})   {len(states)} configuration states  {[d for d, _ in states]}")

for label, atom, N in [('F', 'F 0 0 0', 9), ('F+', 'F 0 0 0', 8), ('O', 'O 0 0 0', 8), ('O-', 'O 0 0 0', 9), ('O+', 'O 0 0 0', 7),
                       ('CO', 'C 0 0 0; O 0 0 2.132', 14), ('CO-', 'C 0 0 0; O 0 0 2.132', 15), ('CO+', 'C 0 0 0; O 0 0 2.132', 13),
                       ('O2', 'O 0 0 0; O 0 0 2.3', 16), ('O2+', 'O 0 0 0; O 0 0 2.3', 15), ('O2-', 'O 0 0 0; O 0 0 2.3', 17)]:
    count(atom, N, label)

F      N= 9  Hund: 5 alpha + 4 beta (2S=1)   3 configuration states  ['b-hole0', 'b-hole1', 'b-hole2']


F+     N= 8  Hund: 5 alpha + 3 beta (2S=2)   3 configuration states  ['b-hole0 b-hole1', 'b-hole0 b-hole2', 'b-hole1 b-hole2']


O      N= 8  Hund: 5 alpha + 3 beta (2S=2)   3 configuration states  ['b-hole0 b-hole1', 'b-hole0 b-hole2', 'b-hole1 b-hole2']
O-     N= 9  Hund: 5 alpha + 4 beta (2S=1)   3 configuration states  ['b-hole0', 'b-hole1', 'b-hole2']


O+     N= 7  Hund: 5 alpha + 2 beta (2S=3)   0 configuration states  []


CO     N=14  Hund: 7 alpha + 7 beta (2S=0)   0 configuration states  []


CO-    N=15  Hund: 8 alpha + 7 beta (2S=1)   2 configuration states  ['a-hole0', 'a-hole1']


CO+    N=13  Hund: 7 alpha + 6 beta (2S=1)   0 configuration states  []


O2     N=16  Hund: 9 alpha + 7 beta (2S=2)   0 configuration states  []


O2+    N=15  Hund: 8 alpha + 7 beta (2S=1)   2 configuration states  ['a-hole0', 'a-hole1']


O2-    N=17  Hund: 9 alpha + 8 beta (2S=1)   2 configuration states  ['b-hole0', 'b-hole1']


Reading the table: `b-hole0 b-hole1` is a state with a hole (population 0) in the first and the second orbital of the beta channel; the numbers are the orbitals of the degenerate
shell. F$^+$ has three states (a beta electron in one of the three 2p orbitals), O$^+$ ($2p^3$, half filled) and closed shells need no ensemble, and the degenerate $\pi^*$ pair of
CO$^-$, O$_2^+$ and O$_2^-$ gives two states. Molecules with degenerate orbitals other than linear ones work the same way, as long as the fragment has
exactly degenerate orbitals: for the benzene cation the hole in the degenerate $e_{1g}$ pair gives two states.

In [5]:
atoms = []
for k in range(6):
    a = np.pi / 3 * k
    atoms += [f"C {2.6355*np.cos(a):.6f} {2.6355*np.sin(a):.6f} 0", f"H {4.6355*np.cos(a):.6f} {4.6355*np.sin(a):.6f} 0"]
benzene = fgto.M(atom='; '.join(atoms), unit='bohr', basis='sto-3g', fragments={0: list(range(12))}, charge=1, spin=1, fcharge={0: 1})
ref = benzene.shells(0, 41)
print(ref)
print([d for d, _ in ref.states(*ref.hund())])

Shells of fragment 0 with 41 electrons (Hund: N_alpha=21, N_beta=20, 2S=1); orbitals in order of energy
  group  degeneracy  energy (Eh)   electrons   main AO components
      0           1     -9.89096       2/2     0 C 1s (0.16), 6 C 1s (0.16)
      1           2     -9.89029       4/4     6 C 1s (0.16), 0 C 1s (0.16)
      2           2     -9.88907       4/4     10 C 1s (0.16), 8 C 1s (0.16)
      3           1     -9.88827       2/2     4 C 1s (0.17), 2 C 1s (0.17)
      4           1     -1.00534       2/2     0 C 2s (0.14), 6 C 2s (0.14)
      5           2     -0.89661       4/4     6 C 2s (0.12), 0 C 2s (0.12)
      6           2     -0.76057       4/4     0 C 2py (0.07), 6 C 2py (0.07)
      7           1     -0.69105       2/2     0 C 2px (0.11), 6 C 2px (0.11)
      8           1     -0.62228       2/2     0 C 2py (0.17), 6 C 2py (0.17)
      9           1     -0.61587       2/2     11 H 1s (0.07), 9 H 1s (0.07)
     10           2     -0.59082       4/4     6 C 2px (0.11),

## Options of `'config'`

```python
ensembles={'config': [0, 1]}                      # fragments 0 and 1
ensembles={'config': 'all'}                       # every fragment that has a degenerate partly filled shell
ensembles={'config': {0: {'fill': 'occupied'}}}   # constraints on occupied orbitals instead of holes
ensembles={'config': {0: {'members': [9]}}}       # only the member with 9 electrons gets the ensemble
ensembles={'config': {0: {'omega': (0.5, 0.3, 0.2)}}}                 # initial weights (equal by default)
ensembles={'config': {0: {'omega': {9: (0.5, 0.3, 0.2)}}}}            # initial weights of the member with 9 electrons
```

The weights are optimized during the P-DFT run (`optimize_omega=False` freezes them). States chosen by hand are also possible:
`{'config': {0: {'charge': 0, 'states': ((('F 2px', 'beta', 0.0),), ...)}}}` (see the README).